# Week 1: Data Profiling & Structure Analysis
**Project:** Heavy Suppliers Warehouse Analytics (Individual Track)
**Goal:** Profile all 12 tables, check how they join, and list data quality issues to fix in Week 2.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")
FILES = ["branches","customers","products","suppliers","inventory_master","stock_ledger",
         "sales_orders_header","sales_orders_lines","purchase_orders_header","purchase_orders_lines",
         "invoices","payments"]
dfs = {n: pd.read_csv(DATA_DIR / f"{n}.csv") for n in FILES}
print({n: d.shape for n, d in dfs.items()})

## 1. Profile summary of every table

In [ ]:
rows = []
for n, d in dfs.items():
    rows.append({"table": n, "rows": len(d), "columns": d.shape[1],
                 "missing_cells": int(d.isnull().sum().sum()),
                 "duplicate_rows": int(d.duplicated().sum()),
                 "numeric_cols": d.select_dtypes("number").shape[1],
                 "text_cols": d.select_dtypes(exclude="number").shape[1]})
profile = pd.DataFrame(rows)
profile.to_csv("profile_summary.csv", index=False)
profile

## 2. Column-level detail (types, nulls, unique values)

In [ ]:
def column_profile(name):
    d = dfs[name]
    return pd.DataFrame({"dtype": d.dtypes.astype(str), "nulls": d.isnull().sum(),
                         "null_%": (d.isnull().mean()*100).round(2), "unique": d.nunique()})
for n in FILES:
    print(f"\n=== {n} ===")
    display(column_profile(n)) if "display" in globals() else print(column_profile(n))

## 3. Numeric statistics

In [ ]:
for n in FILES:
    num = dfs[n].select_dtypes("number")
    if not num.empty:
        print(f"\n=== {n} ===")
        print(num.describe().T[["min","mean","max"]].round(2))

## 4. Date ranges

In [ ]:
date_cols = {"sales_orders_header":["order_date","delivery_date"],
             "purchase_orders_header":["order_date","expected_delivery_date","received_date"],
             "stock_ledger":["movement_date"], "invoices":["invoice_date","due_date"],
             "payments":["payment_date"], "customers":["customer_since","last_purchase_date"],
             "products":["last_purchase_date"]}
for t, cols in date_cols.items():
    for c in cols:
        s = pd.to_datetime(dfs[t][c], errors="coerce")
        print(f"{t}.{c}: {s.min().date()} -> {s.max().date()}  (unparseable: {s.isnull().sum() - dfs[t][c].isnull().sum()})")

## 5. Table relationships (orphan check)
Every foreign key should exist in its parent table. Zero orphans means the join is clean.

In [ ]:
checks = [
 ("sales_orders_lines","so_id","sales_orders_header","so_id"),
 ("sales_orders_lines","product_id","products","product_id"),
 ("sales_orders_header","customer_id","customers","customer_id"),
 ("sales_orders_header","branch_id","branches","branch_id"),
 ("purchase_orders_lines","po_id","purchase_orders_header","po_id"),
 ("purchase_orders_lines","product_id","products","product_id"),
 ("purchase_orders_header","supplier_id","suppliers","supplier_id"),
 ("purchase_orders_header","branch_id","branches","branch_id"),
 ("invoices","so_id","sales_orders_header","so_id"),
 ("payments","invoice_id","invoices","invoice_id"),
 ("stock_ledger","product_id","products","product_id"),
 ("stock_ledger","branch_id","branches","branch_id"),
 ("inventory_master","product_id","products","product_id"),
 ("inventory_master","branch_id","branches","branch_id"),
 ("customers","branch_id","branches","branch_id"),
]
rel = pd.DataFrame([{"child":c,"key":k,"parent":p,"orphans":int((~dfs[c][k].isin(dfs[p][pk])).sum())}
                    for c,k,p,pk in checks])
rel

## 6. Business-rule consistency checks

In [ ]:
so, sol = dfs["sales_orders_header"], dfs["sales_orders_lines"]
inv, pay = dfs["invoices"], dfs["payments"]
inm, po = dfs["inventory_master"], dfs["purchase_orders_header"]

line_sum = sol.groupby("so_id").line_total.sum()
print("Sales header total != sum of lines:", int((line_sum != so.set_index("so_id").total_order_value.reindex(line_sum.index)).sum()))
print("Delivery before order date:", int((pd.to_datetime(so.delivery_date) < pd.to_datetime(so.order_date)).sum()))
print("Order status:", so.order_status.value_counts().to_dict())
print("Invoices vs delivered orders:", len(inv), "vs", (so.order_status=="Delivered").sum())
print("PO status:", po.po_status.value_counts().to_dict())
print("PO missing received_date by status:\n", po[po.received_date.isnull()].po_status.value_counts().to_string())
for name, d, k in [("invoices", inv, "invoice_id"), ("payments", pay, "payment_id"),
                   ("sales_orders_header", so, "so_id"), ("purchase_orders_header", po, "po_id")]:
    print(f"Duplicate {k} in {name}:", int(d[k].duplicated().sum()))
inv_u = inv.drop_duplicates("invoice_id").set_index("invoice_id")
paid = pay.groupby("invoice_id").payment_amount.sum()
print("Invoices with no payment record:", int((~inv_u.index.isin(pay.invoice_id)).sum()))
print("Invoices overpaid (payments > invoice total):", int((paid > inv_u.grand_total.reindex(paid.index) + 1).sum()))
print("Tiny payments (< 100):", int((pay.payment_amount < 100).sum()))
print("Inventory rows where current_stock > max_stock:", int((inm.current_stock > inm.max_stock).sum()), "of", len(inm))
print("Inventory current_stock range:", inm.current_stock.min(), "-", inm.current_stock.max(), "| opening_stock range:", inm.opening_stock.min(), "-", inm.opening_stock.max())
print("Branch capacity stored as text:", dfs["branches"].warehouse_capacity.tolist())

## 7. Stock ledger check

In [ ]:
sl = dfs["stock_ledger"]
print(sl.movement_type.value_counts().to_dict())
print("Zero/negative quantities:", int((sl.quantity <= 0).sum()))
print("Duplicate movement_id:", int(sl.movement_id.duplicated().sum()))
last = sl.sort_values("movement_date").groupby(["product_id","branch_id"]).running_balance.last().rename("ledger_last_balance")
cmp = inm.set_index(["product_id","branch_id"]).join(last)
print("Ledger last balance vs inventory_master.current_stock (first 5):")
print(cmp[["current_stock","ledger_last_balance"]].head())

## 8. Findings (to fix in Week 2)
See `findings_summary.md` for the written summary.